# 04 — Supervisor Proof Suite: Kids-on-Kids, Cross-Age (`Adult ↔ Child`), & Flared Lehenga Ablation
Directly tests and visually + quantitatively proves every single question raised by your FYP supervisor:
1. **Experiment A (Kids-on-Kids):** Unseen Peraan kids outfits on unseen kids photos (`FID, LPIPS, SSIM, CLIP-I`).
2. **Experiment B (Cross-Age Matrix):** `Adult Formal Outfit → Child` AND `Child Outfit → Adult` side-by-side proof grid + CLIP-I transfer fidelity.
3. **Experiment C (Flared Wedding Lehenga Ablation):** Compares `Tight Mask` vs `Our Dilated A-Line Flared Mask + V3 LoRA` on unseen Peraan wedding lehengas!

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q diffusers transformers accelerate peft lpips torchmetrics open-clip-torch clean-fid pillow pandas tqdm


In [ ]:
import json
import sys
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import lpips
import open_clip
from pathlib import Path
from PIL import Image, ImageDraw
from torchvision import transforms
from tqdm.auto import tqdm
from torchmetrics.image import StructuralSimilarityIndexMeasure
from cleanfid import fid
from diffusers import AutoencoderKL, DDIMScheduler, UNet2DConditionModel
import peft.import_utils
from peft import PeftModel

# Bypass torchao conflict in Colab peft
peft.import_utils.is_torchao_available = lambda: False
for mod in list(sys.modules.values()):
    if hasattr(mod, 'is_torchao_available'):
        try:
            mod.is_torchao_available = lambda: False
        except Exception:
            pass

device = 'cuda' if torch.cuda.is_available() else 'cpu'
V3_ROOT = Path('/content/drive/MyDrive/FashionFit_PK_V3_Kids_Formal_Extension')
DATA_DIR = V3_ROOT / '01_KIDS_AND_FORMAL_DATA'
TEST_DIR = DATA_DIR / 'paired_test'
EVAL_DIR = V3_ROOT / '04_CROSS_AGE_AND_LEHENGA_EVAL'
REPORT_DIR = V3_ROOT / '05_SUPERVISOR_REPORT_ARTIFACTS'
for d in [EVAL_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

LORA_CKPT = V3_ROOT / '03_CATVTON_LORA_V3_MULTI_MODE' / 'catvton_v3_universal_lora_r16' / 'step_1200'
BASE_ID = 'booksforcharlie/stable-diffusion-inpainting'

vae = AutoencoderKL.from_pretrained(BASE_ID, subfolder='vae', torch_dtype=torch.float16).to(device).eval()
scheduler = DDIMScheduler.from_pretrained(BASE_ID, subfolder='scheduler')
unet_base = UNet2DConditionModel.from_pretrained(BASE_ID, subfolder='unet', torch_dtype=torch.float16).to(device).eval()
unet_v3 = PeftModel.from_pretrained(unet_base, str(LORA_CKPT)).to(device).eval()

img_tf = transforms.Compose([transforms.Resize((512, 384)), transforms.ToTensor(), transforms.Normalize([0.5], [0.5])])
msk_tf = transforms.Compose([transforms.Resize((512, 384), interpolation=transforms.InterpolationMode.NEAREST), transforms.ToTensor()])

@torch.no_grad()
def infer_vton(unet_m, p_im, g_im, m_im, steps=28, seed=42):
    p = img_tf(p_im).unsqueeze(0).to(device, dtype=torch.float16)
    g = img_tf(g_im).unsqueeze(0).to(device, dtype=torch.float16)
    m = (msk_tf(m_im) > 0.5).float().unsqueeze(0).to(device, dtype=torch.float16)
    p_lat = vae.encode(p).latent_dist.mean * vae.config.scaling_factor
    m_lat = vae.encode(p * (1.0 - m)).latent_dist.mean * vae.config.scaling_factor
    g_lat = vae.encode(g).latent_dist.mean * vae.config.scaling_factor
    msk_lat = F.interpolate(m, size=p_lat.shape[-2:], mode='nearest')
    cond_lat = torch.cat([m_lat, g_lat], dim=-1)
    cond_mask = torch.cat([msk_lat, torch.zeros_like(msk_lat)], dim=-1)
    empty_cond = torch.zeros((1, 77, 768), device=device, dtype=torch.float16)
    lat = torch.randn(cond_lat.shape, generator=torch.Generator(device=device).manual_seed(seed), device=device, dtype=torch.float16)
    scheduler.set_timesteps(steps, device=device)
    for t in scheduler.timesteps:
        np_out = unet_m(torch.cat([lat, cond_mask, cond_lat], dim=1), t, encoder_hidden_states=empty_cond).sample
        lat = scheduler.step(np_out, t, lat).prev_sample
    dec = (vae.decode(lat[..., :p_lat.shape[-1]] / vae.config.scaling_factor).sample / 2 + 0.5).clamp(0, 1)
    out_pil = Image.fromarray((dec[0].permute(1, 2, 0).float().cpu().numpy() * 255).astype(np.uint8))
    return Image.composite(out_pil, p_im.resize((384, 512), Image.BILINEAR), m_im.resize((384, 512), Image.NEAREST))

split_df = pd.read_csv(DATA_DIR / 'v3_paired_split_index.csv')
test_kids = split_df[(split_df['split'] == 'test') & (split_df['domain'] == 'kids_ethnic')]['image_id'].tolist()
test_formal = split_df[(split_df['split'] == 'test') & (split_df['domain'] == 'wedding_formals')]['image_id'].tolist()
print(f'Unseen Test Counts -> Kids: {len(test_kids)} | Wedding Formals/Lehenga: {len(test_formal)}')

# --- EXPERIMENT A: Kids-on-Kids & Wedding Formals Visual + Metric Grid ---
def save_grid(strips, headers, out_path):
    if not strips:
        return
    cols = len(headers)
    grid = Image.new('RGB', (384 * cols, 512 * len(strips) + 48), (20, 20, 20))
    draw = ImageDraw.Draw(grid)
    for i, h in enumerate(headers):
        draw.text((i * 384 + 14, 14), h, fill=(255, 255, 255))
    for r, s in enumerate(strips):
        grid.paste(s, (0, 48 + r * 512))
    grid.save(out_path, quality=92)
    print('Saved:', out_path)

# Exp B: Cross-Age Matrix (Adult Formal -> Child, AND Child Outfit -> Adult)
n_cross = min(4, len(test_kids), len(test_formal))
cross_strips = []
clip_model, _, clip_prep = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai', device=device)
clip_model.eval()
cross_clip_a2c, cross_clip_c2a = [], []

def clip_sim(im1, im2):
    with torch.no_grad():
        c1 = clip_model.encode_image(clip_prep(im1).unsqueeze(0).to(device))
        c2 = clip_model.encode_image(clip_prep(im2).unsqueeze(0).to(device))
        c1 = c1 / c1.norm(dim=-1, keepdim=True)
        c2 = c2 / c2.norm(dim=-1, keepdim=True)
        return float((c1 * c2).sum().item())

for i in range(n_cross):
    k_id, a_id = test_kids[i], test_formal[i]
    k_p = Image.open(TEST_DIR / 'person' / f'{k_id}.jpg').convert('RGB')
    k_g = Image.open(TEST_DIR / 'garment' / f'{k_id}.png').convert('RGB')
    k_m = Image.open(TEST_DIR / 'mask_flared' / f'{k_id}.png').convert('L')
    a_p = Image.open(TEST_DIR / 'person' / f'{a_id}.jpg').convert('RGB')
    a_g = Image.open(TEST_DIR / 'garment' / f'{a_id}.png').convert('RGB')
    a_m = Image.open(TEST_DIR / 'mask_flared' / f'{a_id}.png').convert('L')

    out_a2c = infer_vton(unet_v3, k_p, a_g, k_m)   # Adult outfit onto Child
    out_c2a = infer_vton(unet_v3, a_p, k_g, a_m)   # Child outfit onto Adult
    cross_clip_a2c.append(clip_sim(out_a2c, a_g))
    cross_clip_c2a.append(clip_sim(out_c2a, k_g))

    s = Image.new('RGB', (384 * 4, 512))
    s.paste(k_p.resize((384, 512)), (0, 0))
    s.paste(out_a2c, (384, 0))
    s.paste(a_p.resize((384, 512)), (768, 0))
    s.paste(out_c2a, (1152, 0))
    cross_strips.append(s)

save_grid(cross_strips, ['Target Child', 'Adult Outfit -> Child (V3)', 'Target Adult', 'Child Outfit -> Adult (V3)'], REPORT_DIR / 'expB_cross_age_matrix_grid.jpg')

# Exp C: Flared Lehenga Ablation (Standard Tight Mask vs Our Dilated A-Line Flared Mask)
lehenga_strips = []
for a_id in test_formal[:6]:
    p_im = Image.open(TEST_DIR / 'person' / f'{a_id}.jpg').convert('RGB')
    g_im = Image.open(TEST_DIR / 'garment' / f'{a_id}.png').convert('RGB')
    m_tight = Image.open(TEST_DIR / 'mask' / f'{a_id}.png').convert('L')
    m_flare = Image.open(TEST_DIR / 'mask_flared' / f'{a_id}.png').convert('L')

    out_tight = infer_vton(unet_v3, p_im, g_im, m_tight)
    out_flare = infer_vton(unet_v3, p_im, g_im, m_flare)

    s = Image.new('RGB', (384 * 4, 512))
    s.paste(p_im.resize((384, 512)), (0, 0))
    s.paste(g_im.resize((384, 512)), (384, 0))
    s.paste(out_tight, (768, 0))
    s.paste(out_flare, (1152, 0))
    lehenga_strips.append(s)

save_grid(lehenga_strips, ['Ground Truth Formal', 'Target Lehenga/Formal', 'Tight Mask (Standard)', 'Dilated Flared Mask (Ours)'], REPORT_DIR / 'expC_lehenga_flared_ablation_grid.jpg')

summary = {
    'cross_age_adult_to_child_CLIP_I': round(float(np.mean(cross_clip_a2c)), 4) if cross_clip_a2c else None,
    'cross_age_child_to_adult_CLIP_I': round(float(np.mean(cross_clip_c2a)), 4) if cross_clip_c2a else None,
    'num_test_kids': len(test_kids),
    'num_test_wedding_formals': len(test_formal),
}
print('\n=== V3 SUPERVISOR PROOF SUMMARY ===')
print(json.dumps(summary, indent=2))
(REPORT_DIR / 'v3_supervisor_proof_summary.json').write_text(json.dumps(summary, indent=2))


In [ ]:
# --- EXPERIMENT A + FULL QUANTITATIVE TABLE (Baseline vs V3 Universal LoRA on Kids & Wedding Formals) ---
lpips_fn = lpips.LPIPS(net='alex').to(device).eval()
ssim_fn = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)
tf_01 = transforms.Compose([transforms.Resize((512, 384)), transforms.ToTensor()])

def eval_domain_benchmark(id_list, dom_name):
    gt_d = EVAL_DIR / f'{dom_name}_gt'
    base_d = EVAL_DIR / f'{dom_name}_base'
    v3_d = EVAL_DIR / f'{dom_name}_v3'
    for d in [gt_d, base_d, v3_d]:
        d.mkdir(parents=True, exist_ok=True)
    strips = []
    for sid in tqdm(id_list[:25], desc=f'Benchmarking {dom_name}'):
        p_im = Image.open(TEST_DIR / 'person' / f'{sid}.jpg').convert('RGB')
        g_im = Image.open(TEST_DIR / 'garment' / f'{sid}.png').convert('RGB')
        m_im = Image.open(TEST_DIR / 'mask_flared' / f'{sid}.png').convert('L')
        gt_res = p_im.resize((384, 512), Image.BILINEAR)
        gt_res.save(gt_d / f'{sid}.png')
        out_b = infer_vton(unet_base, p_im, g_im, m_im)
        out_v = infer_vton(unet_v3, p_im, g_im, m_im)
        out_b.save(base_d / f'{sid}.png')
        out_v.save(v3_d / f'{sid}.png')
        if len(strips) < 6:
            s = Image.new('RGB', (384 * 4, 512))
            s.paste(gt_res, (0, 0))
            s.paste(g_im.resize((384, 512)), (384, 0))
            s.paste(out_b, (768, 0))
            s.paste(out_v, (1152, 0))
            strips.append(s)
    save_grid(strips, [f'{dom_name} GT', 'Target Outfit', 'Zero-Shot Baseline', 'FashionFit-PK V3 LoRA (Ours)'], REPORT_DIR / f'expA_{dom_name}_grid.jpg')

    def score_dir(pred_d, label):
        s_vals, l_vals, c_vals = [], [], []
        for gp in sorted(gt_d.glob('*.png')):
            pp = pred_d / gp.name
            if not pp.exists():
                continue
            gi, pi = Image.open(gp).convert('RGB'), Image.open(pp).convert('RGB')
            tg, tp = tf_01(gi).unsqueeze(0).to(device), tf_01(pi).unsqueeze(0).to(device)
            s_vals.append(ssim_fn(tp, tg).item())
            l_vals.append(lpips_fn(tp * 2 - 1, tg * 2 - 1).item())
            c_vals.append(clip_sim(gi, pi))
        fid_v = fid.compute_fid(str(gt_d), str(pred_d)) if len(s_vals) >= 10 else float('nan')
        return {'Domain': dom_name, 'Method': label, 'N': len(s_vals), 'FID': round(float(fid_v), 2), 'LPIPS': round(float(np.mean(l_vals)), 4), 'SSIM': round(float(np.mean(s_vals)), 4), 'CLIP_I': round(float(np.mean(c_vals)), 4)}

    return [score_dir(base_d, 'Zero-Shot Baseline'), score_dir(v3_d, 'FashionFit-PK V3 LoRA (Ours)')]

all_bench = []
if test_kids:
    all_bench.extend(eval_domain_benchmark(test_kids, 'kids_ethnic'))
if test_formal:
    all_bench.extend(eval_domain_benchmark(test_formal, 'wedding_formals'))

bdf = pd.DataFrame(all_bench)
print('\n=== V3 QUANTITATIVE BENCHMARK (UNSEEN KIDS & WEDDING FORMALS) ===')
print(bdf.to_string(index=False))
bdf.to_csv(REPORT_DIR / 'v3_all_benchmarks.csv', index=False)
(REPORT_DIR / 'v3_all_benchmarks.json').write_text(json.dumps(all_bench, indent=2))

tex = [
    r'\begin{table}[t]', r'\centering',
    r'\caption{FashionFit-PK V3 Quantitative Evaluation on Unseen Pakistani Kids Ethnic Wear and Wedding Formals/Lehengas.}',
    r'\begin{tabular}{llcccc}', r'\toprule',
    r'Domain & Method & FID $\downarrow$ & LPIPS $\downarrow$ & SSIM $\uparrow$ & CLIP-I $\uparrow$ \\', r'\midrule'
]
for r in all_bench:
    tex.append(f"{r['Domain'].replace('_',' ')} & {r['Method']} & {r['FID']:.2f} & {r['LPIPS']:.4f} & {r['SSIM']:.4f} & {r['CLIP_I']:.4f} \\\\")
tex += [r'\bottomrule', r'\end{tabular}', r'\end{table}']
(REPORT_DIR / 'v3_supervisor_paper_tables.tex').write_text('\n'.join(tex))
print('Saved v3_supervisor_paper_tables.tex to 05_SUPERVISOR_REPORT_ARTIFACTS/!')
